In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os

for dirname, dirs, files in os.walk("/kaggle/input"):
    print("\nFOLDER:", dirname)
    for file in files[:10]:
        print(" ", file)

In [ ]:
import os
import torch
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"

# 1. GPU check
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

# 2. Check metadata
metadata_path = os.path.join(DATA_DIR, "Data_Entry_2017.csv")
df = pd.read_csv(metadata_path)

print("\nMetadata shape:", df.shape)
print("\nColumns:", df.columns.tolist())

# 3. Check official split files
train_val_path = os.path.join(DATA_DIR, "train_val_list.txt")
test_path = os.path.join(DATA_DIR, "test_list.txt")

with open(train_val_path) as f:
    train_val_files = [x.strip() for x in f if x.strip()]

with open(test_path) as f:
    test_files = [x.strip() for x in f if x.strip()]

print("\nOfficial train/validation list:", len(train_val_files))
print("Official test list:", len(test_files))

# 4. Check all 14 disease labels
labels = [
    "Atelectasis", "Cardiomegaly", "Effusion",
    "Infiltration", "Mass", "Nodule", "Pneumonia",
    "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening",
    "Hernia"
]

print("\nNumber of disease classes:", len(labels))

In [ ]:
import os
import glob
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"

# Find all X-ray images
image_paths = glob.glob(
    DATA_DIR + "/images_*/images/*.png"
)

print("Total image files:", len(image_paths))

# Check duplicate image names
image_names = [os.path.basename(p) for p in image_paths]
print("Unique image names:", len(set(image_names)))

# Check metadata
df = pd.read_csv(DATA_DIR + "/Data_Entry_2017.csv")

available_images = set(image_names)
metadata_images = set(df["Image Index"])

print("Metadata records:", len(df))
print("Images missing from folders:", len(metadata_images - available_images))
print("Images missing from metadata:", len(available_images - metadata_images))

In [ ]:
import os
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"

df = pd.read_csv(os.path.join(DATA_DIR, "Data_Entry_2017.csv"))

with open(os.path.join(DATA_DIR, "train_val_list.txt")) as f:
    train_val_files = set(x.strip() for x in f if x.strip())

with open(os.path.join(DATA_DIR, "test_list.txt")) as f:
    test_files = set(x.strip() for x in f if x.strip())

train_val_patients = set(
    df[df["Image Index"].isin(train_val_files)]["Patient ID"]
)

test_patients = set(
    df[df["Image Index"].isin(test_files)]["Patient ID"]
)

overlap = train_val_patients.intersection(test_patients)

print("Train/validation images:", len(train_val_files))
print("Test images:", len(test_files))
print("Train/validation patients:", len(train_val_patients))
print("Test patients:", len(test_patients))
print("Patients appearing in BOTH splits:", len(overlap))

In [ ]:
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# Filter official training/validation images
train_val_df = df[df["Image Index"].isin(train_val_files)].copy()

# Split by Patient ID, not individual images
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_val_df,
        groups=train_val_df["Patient ID"]
    )
)

train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

test_df = df[df["Image Index"].isin(test_files)].copy()

# Verify patient separation
train_patients = set(train_df["Patient ID"])
val_patients = set(val_df["Patient ID"])
test_patients = set(test_df["Patient ID"])

print("TRAIN images:", len(train_df))
print("VALIDATION images:", len(val_df))
print("TEST images:", len(test_df))

print("\nTRAIN patients:", len(train_patients))
print("VALIDATION patients:", len(val_patients))
print("TEST patients:", len(test_patients))

print("\nTrain-validation overlap:", len(train_patients & val_patients))
print("Train-test overlap:", len(train_patients & test_patients))
print("Validation-test overlap:", len(val_patients & test_patients))

assert len(train_patients & val_patients) == 0
assert len(train_patients & test_patients) == 0
assert len(val_patients & test_patients) == 0

print("\nSPLIT VERIFIED SUCCESSFULLY")

In [ ]:
import os
import glob
import numpy as np
import pandas as pd

# The 14 disease labels
LABELS = [
    "Atelectasis",
    "Cardiomegaly",
    "Effusion",
    "Infiltration",
    "Mass",
    "Nodule",
    "Pneumonia",
    "Pneumothorax",
    "Consolidation",
    "Edema",
    "Emphysema",
    "Fibrosis",
    "Pleural_Thickening",
    "Hernia"
]

# Find every image and create filename -> full path mapping
image_paths = glob.glob(
    DATA_DIR + "/images_*/images/*.png"
)

image_map = {
    os.path.basename(path): path
    for path in image_paths
}

# Convert disease names into 14 binary columns
def encode_labels(label_string):
    present = set(label_string.split("|"))
    return [int(label in present) for label in LABELS]

for split_df in [train_df, val_df, test_df]:
    encoded = np.array(
        split_df["Finding Labels"].apply(encode_labels).tolist()
    )

    for i, label in enumerate(LABELS):
        split_df[label] = encoded[:, i]

    split_df["Image Path"] = split_df["Image Index"].map(image_map)

# Verify image paths
print("Total image paths:", len(image_map))
print("Missing training paths:", train_df["Image Path"].isna().sum())
print("Missing validation paths:", val_df["Image Path"].isna().sum())
print("Missing testing paths:", test_df["Image Path"].isna().sum())

# Show disease distribution
print("\nTRAIN disease counts:")
print(train_df[LABELS].sum().sort_values(ascending=False))

print("\nVALIDATION disease counts:")
print(val_df[LABELS].sum().sort_values(ascending=False))

print("\nTEST disease counts:")
print(test_df[LABELS].sum().sort_values(ascending=False))

# Save split files
os.makedirs("/kaggle/working/splits", exist_ok=True)

train_df.to_csv("/kaggle/working/splits/train.csv", index=False)
val_df.to_csv("/kaggle/working/splits/validation.csv", index=False)
test_df.to_csv("/kaggle/working/splits/test.csv", index=False)

print("\nLABEL AND IMAGE PREPARATION COMPLETE")

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("GPU count:", torch.cuda.device_count())

# Image transformations
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Custom Dataset
class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

        self.image_paths = self.dataframe["Image Path"].values
        self.labels = self.dataframe[LABELS].values.astype(np.float32)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(self.labels[index], dtype=torch.float32)

        return image, label


# Create datasets
train_dataset = ChestXrayDataset(train_df, train_transform)
val_dataset = ChestXrayDataset(val_df, eval_transform)
test_dataset = ChestXrayDataset(test_df, eval_transform)

# DataLoaders
BATCH_SIZE = 64
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

print("\nDataset sizes:")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))

# Test one batch
images, labels = next(iter(train_loader))

print("\nBatch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Image data type:", images.dtype)
print("Label data type:", labels.dtype)

print("\nDataLoaders ready!")

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Use pretrained ResNet18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Replace final classification layer with 14 outputs
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(LABELS))

# Move model to GPU
model = model.to(device)

# Use both GPUs
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

# Loss function with moderate class-imbalance weighting
train_labels = train_df[LABELS].values

positive = train_labels.sum(axis=0)
negative = len(train_labels) - positive

pos_weights = np.sqrt(negative / np.maximum(positive, 1))
pos_weights = np.clip(pos_weights, 1.0, 10.0)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=device
)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

# Mixed precision
scaler = torch.amp.GradScaler("cuda")

print("MODEL READY")
print("Architecture: ResNet18")
print("Pretrained: ImageNet")
print("Output classes:", len(LABELS))
print("GPUs used:", torch.cuda.device_count())
print("Batch size:", BATCH_SIZE)
print("Loss: Weighted BCEWithLogitsLoss")
print("Optimizer: AdamW")

print("\nClass weights:")
for label, weight in zip(LABELS, pos_weights.cpu().numpy()):
    print(f"{label}: {weight:.2f}")

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score

# Training configuration
EPOCHS = 10
PATIENCE = 3

SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(SAVE_DIR, "best_resnet18.pth")

# Performance settings
torch.backends.cudnn.benchmark = True

best_auc = 0.0
epochs_without_improvement = 0
history = []

# Helper: get model state without DataParallel wrapper
def get_model_state():
    if isinstance(model, nn.DataParallel):
        return model.module.state_dict()
    return model.state_dict()


# Training loop
for epoch in range(EPOCHS):

    start_time = time.time()

    print(f"\n{'='*60}")
    print(f"EPOCH {epoch + 1}/{EPOCHS}")
    print(f"{'='*60}")

    # -------------------------
    # TRAINING
    # -------------------------
    model.train()

    train_loss = 0.0
    train_samples = 0

    train_bar = tqdm(train_loader, desc="Training")

    for images, labels in train_bar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item() * images.size(0)
        train_samples += images.size(0)

        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = train_loss / train_samples

    # -------------------------
    # VALIDATION
    # -------------------------
    model.eval()

    val_loss = 0.0
    val_samples = 0

    all_predictions = []
    all_targets = []

    with torch.no_grad():

        val_bar = tqdm(val_loader, desc="Validation")

        for images, labels in val_bar:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                outputs = model(images)
                loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_samples += images.size(0)

            probabilities = torch.sigmoid(outputs)

            all_predictions.append(probabilities.float().cpu().numpy())
            all_targets.append(labels.cpu().numpy())

    avg_val_loss = val_loss / val_samples

    y_true = np.concatenate(all_targets)
    y_pred = np.concatenate(all_predictions)

    # Per-disease ROC-AUC
    disease_aucs = {}

    for i, disease in enumerate(LABELS):

        if len(np.unique(y_true[:, i])) == 2:
            disease_aucs[disease] = roc_auc_score(
                y_true[:, i],
                y_pred[:, i]
            )
        else:
            disease_aucs[disease] = np.nan

    macro_auc = np.nanmean(list(disease_aucs.values()))

    micro_auc = roc_auc_score(
        y_true.ravel(),
        y_pred.ravel()
    )

    # Scheduler uses validation macro AUC
    scheduler.step(macro_auc)

    elapsed = (time.time() - start_time) / 60

    # Print results
    print(f"\nEpoch: {epoch + 1}/{EPOCHS}")
    print(f"Training Loss:   {avg_train_loss:.4f}")
    print(f"Validation Loss: {avg_val_loss:.4f}")
    print(f"Macro ROC-AUC:   {macro_auc:.4f}")
    print(f"Micro ROC-AUC:   {micro_auc:.4f}")
    print(f"Time:            {elapsed:.2f} minutes")
    print(f"Learning Rate:   {optimizer.param_groups[0]['lr']:.7f}")

    print("\nPer-disease ROC-AUC:")

    for disease, auc in disease_aucs.items():
        print(f"{disease:22s}: {auc:.4f}")

    # Save epoch history
    history.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
        "macro_auc": macro_auc,
        "micro_auc": micro_auc,
        "time_minutes": elapsed
    })

    pd.DataFrame(history).to_csv(
        "/kaggle/working/training_history.csv",
        index=False
    )

    # Save best model
    if macro_auc > best_auc:

        best_auc = macro_auc
        epochs_without_improvement = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": get_model_state(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_macro_auc": best_auc,
            "labels": LABELS
        }, BEST_MODEL_PATH)

        print("\nBEST MODEL SAVED!")
        print(f"Best Macro AUC: {best_auc:.4f}")

    else:
        epochs_without_improvement += 1
        print(
            f"\nNo improvement: "
            f"{epochs_without_improvement}/{PATIENCE}"
        )

    # Early stopping
    if epochs_without_improvement >= PATIENCE:
        print("\nEARLY STOPPING TRIGGERED")
        break

print("\n" + "="*60)
print("TRAINING FINISHED")
print("Best Validation Macro AUC:", round(best_auc, 4))
print("Best model:", BEST_MODEL_PATH)
print("="*60)

In [ ]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_name(i))

In [ ]:
import os
import sys
import importlib.util
import importlib.metadata as metadata

print("Python:", sys.version)

try:
    print("PyTorch package version:", metadata.version("torch"))
except Exception as e:
    print("Version error:", e)

spec = importlib.util.find_spec("torch")
print("\nPyTorch location:", spec.origin)

torch_dir = os.path.dirname(spec.origin)

print("\nChecking important folders:")

for folder in ["autograd", "jit", "cuda", "nn"]:
    path = os.path.join(torch_dir, folder)
    print(folder, "exists:", os.path.isdir(path))

print("\nChecking autograd files:")

for filename in ["__init__.py", "grad_mode.py", "function.py"]:
    path = os.path.join(torch_dir, "autograd", filename)
    print(filename, "exists:", os.path.isfile(path))

print("\nInstalled packages:")
for package in ["torch", "torchvision", "torchaudio"]:
    try:
        print(package, metadata.version(package))
    except Exception:
        print(package, "not installed")

In [ ]:
!pip install --no-cache-dir --upgrade \
    torch==2.12.1 \
    torchvision==0.27.1 \
    torchaudio==2.12.1 \
    --index-url https://download.pytorch.org/whl/cu130

In [ ]:
!pip install --no-cache-dir --upgrade \
    torch==2.12.1 \
    torchvision==0.27.1 \
    --index-url https://download.pytorch.org/whl/cu130

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print("GPU", i, torch.cuda.get_device_name(i))

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torchvision
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

print("All libraries imported successfully!")
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
DATA_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"

df = pd.read_csv(
    os.path.join(DATA_DIR, "Data_Entry_2017.csv")
)

print("Dataset loaded:", df.shape)
print("Columns:", df.columns.tolist())

train_val_files = set(
    open(os.path.join(DATA_DIR, "train_val_list.txt")).read().splitlines()
)

test_files = set(
    open(os.path.join(DATA_DIR, "test_list.txt")).read().splitlines()
)

print("Official train/validation images:", len(train_val_files))
print("Official test images:", len(test_files))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

train_val_df = df[df["Image Index"].isin(train_val_files)].copy()
test_df = df[df["Image Index"].isin(test_files)].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_val_df,
        groups=train_val_df["Patient ID"]
    )
)

train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

print("TRAIN images:", len(train_df))
print("VALIDATION images:", len(val_df))
print("TEST images:", len(test_df))

print("\nPatient overlap:")
print("Train-validation:", len(set(train_df["Patient ID"]) & set(val_df["Patient ID"])))
print("Train-test:", len(set(train_df["Patient ID"]) & set(test_df["Patient ID"])))
print("Validation-test:", len(set(val_df["Patient ID"]) & set(test_df["Patient ID"])))

In [ ]:
LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion",
    "Infiltration", "Mass", "Nodule", "Pneumonia",
    "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening",
    "Hernia"
]

# Find all X-ray images
image_paths = glob.glob(
    DATA_DIR + "/images_*/images/*.png"
)

image_map = {
    os.path.basename(path): path
    for path in image_paths
}

# Convert disease labels into 0/1 values
def encode_labels(label_string):
    present = set(label_string.split("|"))
    return [int(label in present) for label in LABELS]

# Add labels and image paths to each split
for split_df in [train_df, val_df, test_df]:
    encoded = np.array(
        split_df["Finding Labels"].apply(encode_labels).tolist()
    )

    for i, label in enumerate(LABELS):
        split_df[label] = encoded[:, i]

    split_df["Image Path"] = split_df["Image Index"].map(image_map)

print("Total images found:", len(image_paths))
print("Training missing paths:", train_df["Image Path"].isna().sum())
print("Validation missing paths:", val_df["Image Path"].isna().sum())
print("Testing missing paths:", test_df["Image Path"].isna().sum())

print("\nDisease classes:", len(LABELS))
print("Labels:", LABELS)

In [ ]:
# Image preprocessing
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# Custom dataset
class ChestXrayDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

        self.image_paths = self.dataframe["Image Path"].values
        self.labels = self.dataframe[LABELS].values.astype(np.float32)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image = Image.open(self.image_paths[index]).convert("RGB")

        if self.transform:
            image = self.transform(image)

        label = torch.tensor(
            self.labels[index],
            dtype=torch.float32
        )

        return image, label


# Create datasets
train_dataset = ChestXrayDataset(train_df, train_transform)
val_dataset = ChestXrayDataset(val_df, eval_transform)
test_dataset = ChestXrayDataset(test_df, eval_transform)


# DataLoader configuration
BATCH_SIZE = 64
NUM_WORKERS = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


# Test one batch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

images, labels = next(iter(train_loader))

print("Device:", device)
print("GPUs:", torch.cuda.device_count())
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Testing images:", len(test_dataset))
print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("DataLoaders ready!")

In [ ]:
# Load pretrained ResNet18
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)

# Replace final layer for 14 disease classes
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(LABELS))

# Move model to GPU
model = model.to(device)

# Use both GPUs
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

# Calculate class weights for imbalanced data
train_labels = train_df[LABELS].values

positive = train_labels.sum(axis=0)
negative = len(train_labels) - positive

pos_weights = np.sqrt(
    negative / np.maximum(positive, 1)
)

pos_weights = np.clip(pos_weights, 1.0, 10.0)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float32,
    device=device
)

# Loss function
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weights
)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.0001
)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

# Mixed precision training
scaler = torch.amp.GradScaler("cuda")

print("MODEL READY")
print("Architecture: ResNet18")
print("Pretrained: ImageNet")
print("Output classes:", len(LABELS))
print("GPUs used:", torch.cuda.device_count())
print("Batch size:", BATCH_SIZE)
print("Loss: Weighted BCEWithLogitsLoss")
print("Optimizer: AdamW")

for label, weight in zip(LABELS, pos_weights.cpu().numpy()):
    print(f"{label}: {weight:.2f}")

In [ ]:
import time
import copy
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

# Training configuration
EPOCHS = 10
PATIENCE = 3

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

best_auc = 0.0
early_stop_counter = 0
history = []

print("TRAINING STARTED")
print("Epochs:", EPOCHS)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("GPUs:", torch.cuda.device_count())

for epoch in range(EPOCHS):

    start_time = time.time()

    # ---------------- TRAINING ----------------
    model.train()
    running_loss = 0.0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS} - Training"
    )

    for images, labels in train_bar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_dataset)

    # ---------------- VALIDATION ----------------
    model.eval()

    val_loss_total = 0.0
    all_targets = []
    all_probabilities = []

    with torch.no_grad():

        for images, labels in tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{EPOCHS} - Validation"
        ):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)

            probabilities = torch.sigmoid(outputs)

            val_loss_total += loss.item() * images.size(0)

            all_targets.append(labels.cpu().numpy())
            all_probabilities.append(probabilities.cpu().numpy())

    val_loss = val_loss_total / len(val_dataset)

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probabilities)

    # Calculate AUC for each disease
    class_aucs = []

    for i, label in enumerate(LABELS):
        if len(np.unique(y_true[:, i])) == 2:
            auc = roc_auc_score(y_true[:, i], y_prob[:, i])
            class_aucs.append(auc)
            print(f"{label}: AUC = {auc:.4f}")

    macro_auc = np.mean(class_aucs)
    micro_auc = roc_auc_score(
        y_true.ravel(),
        y_prob.ravel()
    )

    scheduler.step(macro_auc)

    epoch_time = (time.time() - start_time) / 60

    print("\n" + "=" * 55)
    print(f"EPOCH {epoch+1}/{EPOCHS}")
    print(f"Training Loss:   {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Macro AUC:       {macro_auc:.4f}")
    print(f"Micro AUC:       {micro_auc:.4f}")
    print(f"Time:            {epoch_time:.2f} minutes")
    print("=" * 55)

    history.append({
        "Epoch": epoch + 1,
        "Train Loss": train_loss,
        "Validation Loss": val_loss,
        "Macro AUC": macro_auc,
        "Micro AUC": micro_auc,
        "Time (minutes)": epoch_time
    })

    # Save best model
    if macro_auc > best_auc:

        best_auc = macro_auc
        early_stop_counter = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "macro_auc": macro_auc,
            "micro_auc": micro_auc,
            "labels": LABELS
        }, "/kaggle/working/checkpoints/best_resnet18.pth")

        print("BEST MODEL SAVED!")

    else:
        early_stop_counter += 1
        print(f"No improvement: {early_stop_counter}/{PATIENCE}")

    # Save training history
    pd.DataFrame(history).to_csv(
        "/kaggle/working/training_history.csv",
        index=False
    )

    if early_stop_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

print("\nTRAINING FINISHED")
print("Best Validation Macro AUC:", best_auc)
print("Best model: /kaggle/working/checkpoints/best_resnet18.pth")

In [ ]:
# Bypass the cuDNN library causing the error
torch.backends.cudnn.enabled = False

# Clear GPU cache
torch.cuda.empty_cache()

# Test one training batch
model.train()

images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

with torch.autocast(device_type="cuda", dtype=torch.float16):
    outputs = model(images)
    loss = criterion(outputs, labels)

print("Forward pass successful!")
print("Output shape:", outputs.shape)
print("Loss:", loss.item())
print("GPU count:", torch.cuda.device_count())

In [ ]:
optimizer.zero_grad(set_to_none=True)

scaler.scale(loss).backward()
scaler.step(optimizer)
scaler.update()

print("Forward pass: SUCCESS")
print("Backward pass: SUCCESS")
print("Optimizer step: SUCCESS")


In [ ]:
import time

EPOCHS = 10
PATIENCE = 3

best_auc = 0.0
early_stop_counter = 0
history = []

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

for epoch in range(EPOCHS):

    start_time = time.time()

    # TRAINING
    model.train()
    running_loss = 0.0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Training")

    for images, labels in train_bar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_dataset)

    # VALIDATION
    model.eval()
    val_loss_total = 0.0
    all_targets = []
    all_probabilities = []

    with torch.no_grad():

        for images, labels in tqdm(
            val_loader,
            desc=f"Epoch {epoch+1}/{EPOCHS} Validation"
        ):

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(images)
                loss = criterion(outputs, labels)

            probabilities = torch.sigmoid(outputs)

            val_loss_total += loss.item() * images.size(0)
            all_targets.append(labels.cpu().numpy())
            all_probabilities.append(probabilities.cpu().numpy())

    val_loss = val_loss_total / len(val_dataset)

    y_true = np.concatenate(all_targets)
    y_prob = np.concatenate(all_probabilities)

    # AUC METRICS
    class_aucs = []

    for i, label in enumerate(LABELS):
        if len(np.unique(y_true[:, i])) == 2:
            auc = roc_auc_score(y_true[:, i], y_prob[:, i])
            class_aucs.append(auc)
            print(f"{label}: AUC = {auc:.4f}")

    macro_auc = np.mean(class_aucs)
    micro_auc = roc_auc_score(y_true.ravel(), y_prob.ravel())

    scheduler.step(macro_auc)

    epoch_time = (time.time() - start_time) / 60

    print("\n" + "=" * 45)
    print(f"EPOCH {epoch+1}/{EPOCHS}")
    print(f"Training Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Macro AUC: {macro_auc:.4f}")
    print(f"Micro AUC: {micro_auc:.4f}")
    print(f"Time: {epoch_time:.2f} minutes")
    print("=" * 45)

    history.append({
        "Epoch": epoch + 1,
        "Train Loss": train_loss,
        "Validation Loss": val_loss,
        "Macro AUC": macro_auc,
        "Micro AUC": micro_auc,
        "Time (minutes)": epoch_time
    })

    # SAVE BEST MODEL
    if macro_auc > best_auc:

        best_auc = macro_auc
        early_stop_counter = 0

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "macro_auc": macro_auc,
            "micro_auc": micro_auc,
            "labels": LABELS
        }, "/kaggle/working/checkpoints/best_resnet18.pth")

        print("BEST MODEL SAVED!")

    else:
        early_stop_counter += 1
        print(f"No improvement: {early_stop_counter}/{PATIENCE}")

    pd.DataFrame(history).to_csv(
        "/kaggle/working/training_history.csv",
        index=False
    )

    if early_stop_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

print("\nTRAINING FINISHED")
print("Best validation Macro AUC:", best_auc)

In [ ]:
# Load the best checkpoint (Epoch 2)
checkpoint = torch.load(
    "/kaggle/working/checkpoints/best_resnet18.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Loaded best model from Epoch:", checkpoint["epoch"])
print("Best validation Macro AUC:", checkpoint["macro_auc"])

# Test evaluation
test_targets = []
test_probabilities = []

test_loss_total = 0.0

with torch.no_grad():

    for images, labels in tqdm(test_loader, desc="Testing ResNet18"):

        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        probabilities = torch.sigmoid(outputs)

        test_loss_total += loss.item() * images.size(0)

        test_targets.append(labels.cpu().numpy())
        test_probabilities.append(probabilities.cpu().numpy())

# Combine predictions
y_test = np.concatenate(test_targets)
y_test_prob = np.concatenate(test_probabilities)

test_loss = test_loss_total / len(test_dataset)

# Calculate disease-wise AUC
test_aucs = []

print("\nDISEASE-WISE TEST AUC")
print("-" * 40)

for i, label in enumerate(LABELS):

    auc = roc_auc_score(y_test[:, i], y_test_prob[:, i])
    test_aucs.append(auc)

    print(f"{label}: {auc:.4f}")

macro_test_auc = np.mean(test_aucs)

micro_test_auc = roc_auc_score(
    y_test.ravel(),
    y_test_prob.ravel()
)

print("\n" + "=" * 45)
print("FINAL RESNET18 TEST RESULTS")
print("=" * 45)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Macro AUC: {macro_test_auc:.4f}")
print(f"Test Micro AUC: {micro_test_auc:.4f}")
print("=" * 45)

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

# 1. Generate validation predictions using best model
model.eval()

val_targets = []
val_probabilities = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Validation predictions"):

        images = images.to(device)
        labels = labels.to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)

        probabilities = torch.sigmoid(outputs)

        val_targets.append(labels.cpu().numpy())
        val_probabilities.append(probabilities.cpu().numpy())

y_val = np.concatenate(val_targets)
y_val_prob = np.concatenate(val_probabilities)

# 2. Select best F1 threshold for each disease using validation
thresholds = []

for i, label in enumerate(LABELS):

    precision, recall, threshold_values = precision_recall_curve(
        y_val[:, i],
        y_val_prob[:, i]
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1] /
        (precision[:-1] + recall[:-1] + 1e-8)
    )

    best_threshold = threshold_values[np.argmax(f1_values)]
    thresholds.append(best_threshold)

    print(f"{label}: Threshold = {best_threshold:.4f}")

thresholds = np.array(thresholds)

# 3. Apply validation thresholds to test predictions
y_pred = (
    y_test_prob >= thresholds.reshape(1, -1)
).astype(int)

# 4. Disease-wise metrics
results = []

for i, label in enumerate(LABELS):

    tn, fp, fn, tp = confusion_matrix(
        y_test[:, i],
        y_pred[:, i],
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_test[:, i], y_pred[:, i], zero_division=0
    )

    recall = recall_score(
        y_test[:, i], y_pred[:, i], zero_division=0
    )

    f1 = f1_score(
        y_test[:, i], y_pred[:, i], zero_division=0
    )

    specificity = tn / (tn + fp + 1e-8)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    results.append({
        "Disease": label,
        "Threshold": thresholds[i],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall/Sensitivity": recall,
        "Specificity": specificity,
        "F1-score": f1
    })

results_df = pd.DataFrame(results)

print("\nDISEASE-WISE TEST METRICS")
display(results_df.round(4))

print("\nOVERALL METRICS")
print("Macro Precision:", precision_score(
    y_test, y_pred, average="macro", zero_division=0
))
print("Macro Recall:", recall_score(
    y_test, y_pred, average="macro", zero_division=0
))
print("Macro F1:", f1_score(
    y_test, y_pred, average="macro", zero_division=0
))
print("Micro F1:", f1_score(
    y_test, y_pred, average="micro", zero_division=0
))
print("Exact-match accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

# 1. Generate validation predictions using best model
model.eval()

val_targets = []
val_probabilities = []

with torch.no_grad():
    for images, labels in tqdm(val_loader, desc="Validation predictions"):

        images = images.to(device)
        labels = labels.to(device)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)

        probabilities = torch.sigmoid(outputs)

        val_targets.append(labels.cpu().numpy())
        val_probabilities.append(probabilities.cpu().numpy())

y_val = np.concatenate(val_targets)
y_val_prob = np.concatenate(val_probabilities)

# 2. Select best F1 threshold for each disease using validation
thresholds = []

for i, label in enumerate(LABELS):

    precision, recall, threshold_values = precision_recall_curve(
        y_val[:, i],
        y_val_prob[:, i]
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1] /
        (precision[:-1] + recall[:-1] + 1e-8)
    )

    best_threshold = threshold_values[np.argmax(f1_values)]
    thresholds.append(best_threshold)

    print(f"{label}: Threshold = {best_threshold:.4f}")

thresholds = np.array(thresholds)

# 3. Apply validation thresholds to test predictions
y_pred = (
    y_test_prob >= thresholds.reshape(1, -1)
).astype(int)

# 4. Disease-wise metrics
results = []

for i, label in enumerate(LABELS):

    tn, fp, fn, tp = confusion_matrix(
        y_test[:, i],
        y_pred[:, i],
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_test[:, i], y_pred[:, i], zero_division=0
    )

    recall = recall_score(
        y_test[:, i], y_pred[:, i], zero_division=0
    )

    f1 = f1_score(
        y_test[:, i], y_pred[:, i], zero_division=0
    )

    specificity = tn / (tn + fp + 1e-8)
    accuracy = (tp + tn) / (tp + tn + fp + fn)

    results.append({
        "Disease": label,
        "Threshold": thresholds[i],
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall/Sensitivity": recall,
        "Specificity": specificity,
        "F1-score": f1
    })

results_df = pd.DataFrame(results)

print("\nDISEASE-WISE TEST METRICS")
display(results_df.round(4))

print("\nOVERALL METRICS")
print("Macro Precision:", precision_score(
    y_test, y_pred, average="macro", zero_division=0
))
print("Macro Recall:", recall_score(
    y_test, y_pred, average="macro", zero_division=0
))
print("Macro F1:", f1_score(
    y_test, y_pred, average="macro", zero_division=0
))
print("Micro F1:", f1_score(
    y_test, y_pred, average="micro", zero_division=0
))
print("Exact-match accuracy:", accuracy_score(y_test, y_pred))

In [ ]:
# Recreate ResNet18 architecture
model = models.resnet18(weights=None)

# Restore 14 output classes
model.fc = nn.Linear(model.fc.in_features, len(LABELS))

# Move to GPU
model = model.to(device)

# Restore two-GPU configuration
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

# Load saved best checkpoint
checkpoint = torch.load(
    "/kaggle/working/checkpoints/best_resnet18.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Keep the cuDNN workaround
torch.backends.cudnn.enabled = False

print("MODEL RESTORED SUCCESSFULLY")
print("Best epoch:", checkpoint["epoch"])
print("Validation Macro AUC:", checkpoint["macro_auc"])
print("GPU count:", torch.cuda.device_count())

In [ ]:
# 1. Restore imports
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix,
    roc_auc_score
)
from tqdm.auto import tqdm

# 2. GPU configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.enabled = False

# 3. Restore ResNet18
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 14)
model = model.to(device)

if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

# 4. Load saved checkpoint
checkpoint = torch.load(
    "/kaggle/working/checkpoints/best_resnet18.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("MODEL RESTORED SUCCESSFULLY")
print("Best epoch:", checkpoint["epoch"])
print("Validation Macro AUC:", checkpoint["macro_auc"])
print("GPU count:", torch.cuda.device_count())

In [ ]:
import os

print("CHECKPOINT FILES:")
for root, dirs, files in os.walk("/kaggle"):
    dirs[:] = [d for d in dirs if d not in ["input", ".cache"]]
    for file in files:
        if file.endswith((".pth", ".pt", ".ckpt")):
            print(os.path.join(root, file))

In [ ]:
import os

for path in [
    "/kaggle/working",
    "/kaggle/working/checkpoints",
    "/kaggle/input"
]:
    print("\n", path)
    if os.path.exists(path):
        print(os.listdir(path)[:30])
    else:
        print("NOT FOUND")

In [ ]:
import torch
import os

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

DATA_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"

print("\nDataset exists:", os.path.exists(DATA_DIR))
print("Dataset folders:", os.listdir(DATA_DIR)[:10])

In [ ]:
import os
import glob
import time
import copy
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

# GPU setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.enabled = False

# Dataset paths
DATA_DIR = "/kaggle/input/datasets/nih-chest-xrays/data"
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# 14 disease labels
LABELS = [
    "Atelectasis", "Cardiomegaly", "Effusion",
    "Infiltration", "Mass", "Nodule", "Pneumonia",
    "Pneumothorax", "Consolidation", "Edema",
    "Emphysema", "Fibrosis", "Pleural_Thickening",
    "Hernia"
]

print("Environment ready")
print("Device:", device)
print("GPUs:", torch.cuda.device_count())
print("Checkpoint folder:", CHECKPOINT_DIR)
print("Number of labels:", len(LABELS))

In [ ]:
# Load metadata
metadata = pd.read_csv(
    os.path.join(DATA_DIR, "Data_Entry_2017.csv")
)

# Remove leading/trailing spaces from column names
metadata.columns = metadata.columns.str.strip()

# Find all image files
image_files = glob.glob(
    DATA_DIR + "/images_*/images/*.png"
)

image_map = {
    os.path.basename(path): path
    for path in image_files
}

# Keep images that actually exist
metadata = metadata[
    metadata["Image Index"].isin(image_map)
].copy()

# Encode 14 diseases as binary labels
for label in LABELS:
    metadata[label] = metadata["Finding Labels"].apply(
        lambda x: int(label in x.split("|"))
    )

print("Metadata shape:", metadata.shape)
print("Images found:", len(image_map))
print("Missing images:", (~metadata["Image Index"].isin(image_map)).sum())
print("\nDisease distribution:")
print(metadata[LABELS].sum())

In [ ]:
# Load official test filenames
test_file = os.path.join(DATA_DIR, "test_list.txt")

with open(test_file, "r") as f:
    official_test = set(line.strip() for line in f)

# Separate official test set
test_df = metadata[
    metadata["Image Index"].isin(official_test)
].copy()

# Remaining data for train/validation
train_val_df = metadata[
    ~metadata["Image Index"].isin(official_test)
].copy()

# Patient-level split: prevent patient overlap
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_val_df,
        groups=train_val_df["Patient ID"]
    )
)

train_df = train_val_df.iloc[train_idx].copy()
val_df = train_val_df.iloc[val_idx].copy()

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

# Verify patient overlap
train_patients = set(train_df["Patient ID"])
val_patients = set(val_df["Patient ID"])
test_patients = set(test_df["Patient ID"])

print("\nPatient overlap:")
print("Train-Validation:", len(train_patients & val_patients))
print("Train-Test:", len(train_patients & test_patients))
print("Validation-Test:", len(val_patients & test_patients))

In [ ]:
# Image transformations
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class ChestXrayDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = image_map[row["Image Index"]]
        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        labels = torch.tensor(
            row[LABELS].values.astype(np.float32),
            dtype=torch.float32
        )

        return image, labels


train_dataset = ChestXrayDataset(train_df, train_transform)
val_dataset = ChestXrayDataset(val_df, eval_transform)
test_dataset = ChestXrayDataset(test_df, eval_transform)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Testing dataset:", len(test_dataset))

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    persistent_workers=True
)

# Verify one batch
images, labels = next(iter(train_loader))

print("Image batch:", images.shape)
print("Label batch:", labels.shape)
print("Batch size:", BATCH_SIZE)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

In [ ]:
# Calculate class weights for imbalanced diseases
positive = train_df[LABELS].sum().values
negative = len(train_df) - positive

pos_weight = np.sqrt(negative / positive)
pos_weight = np.clip(pos_weight, 1, 10)

pos_weight = torch.tensor(
    pos_weight,
    dtype=torch.float32,
    device=device
)

# Load ImageNet-pretrained ResNet18
model = models.resnet18(
    weights=models.ResNet18_Weights.DEFAULT
)

# Replace final layer for 14 disease predictions
model.fc = nn.Linear(model.fc.in_features, 14)

model = model.to(device)

# Use both Tesla T4 GPUs
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

# Loss and optimizer
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

# Mixed precision
scaler = torch.amp.GradScaler("cuda")

print("Model:", "ResNet18")
print("Output classes:", model.module.fc.out_features if isinstance(model, nn.DataParallel) else model.fc.out_features)
print("GPUs used:", torch.cuda.device_count())
print("Loss: Weighted BCEWithLogitsLoss")
print("Optimizer: AdamW")
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Checkpoint directory:", CHECKPOINT_DIR)

In [ ]:
from sklearn.metrics import roc_auc_score

EPOCHS = 10
PATIENCE = 3

best_auc = 0.0
epochs_without_improvement = 0
history = []

best_path = os.path.join(CHECKPOINT_DIR, "best_resnet18.pth")
latest_path = os.path.join(CHECKPOINT_DIR, "latest_resnet18.pth")

for epoch in range(EPOCHS):

    start_time = time.time()

    # ---------------- TRAINING ----------------
    model.train()
    train_loss = 0.0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS} - Training"
    )

    for images, targets in train_bar:

        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= len(train_loader)

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss = 0.0
    all_targets = []
    all_probs = []

    with torch.no_grad():

        for images, targets in tqdm(
            val_loader,
            desc="Validation"
        ):

            images = images.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, targets)

            probabilities = torch.sigmoid(outputs)

            val_loss += loss.item()
            all_targets.append(targets.cpu().numpy())
            all_probs.append(probabilities.cpu().numpy())

    val_loss /= len(val_loader)

    y_val = np.concatenate(all_targets)
    y_val_prob = np.concatenate(all_probs)

    # Macro and micro AUC
    disease_aucs = []

    for i in range(len(LABELS)):
        if len(np.unique(y_val[:, i])) == 2:
            disease_aucs.append(
                roc_auc_score(y_val[:, i], y_val_prob[:, i])
            )

    macro_auc = np.mean(disease_aucs)
    micro_auc = roc_auc_score(
        y_val.ravel(),
        y_val_prob.ravel()
    )

    scheduler.step(macro_auc)

    epoch_time = (time.time() - start_time) / 60

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Macro AUC: {macro_auc:.4f}")
    print(f"Validation Micro AUC: {micro_auc:.4f}")
    print(f"Time: {epoch_time:.2f} minutes")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_auc": macro_auc,
        "micro_auc": micro_auc,
        "time_minutes": epoch_time
    })

    # Save latest checkpoint after EVERY epoch
    checkpoint_data = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "macro_auc": macro_auc,
        "history": history
    }

    torch.save(checkpoint_data, latest_path)

    # Save best checkpoint
    if macro_auc > best_auc:
        best_auc = macro_auc
        epochs_without_improvement = 0

        torch.save(checkpoint_data, best_path)

        print("BEST CHECKPOINT SAVED")
    else:
        epochs_without_improvement += 1

    pd.DataFrame(history).to_csv(
        os.path.join(CHECKPOINT_DIR, "training_history.csv"),
        index=False
    )

    print("Latest checkpoint saved.")
    print("Best validation Macro AUC:", round(best_auc, 4))

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nTRAINING COMPLETED")
print("Best Macro AUC:", best_auc)
print("Best checkpoint:", best_path)
print("Latest checkpoint:", latest_path)

In [ ]:
import shutil
from IPython.display import display, FileLink

# Package best model, latest model and training history
shutil.make_archive(
    "/kaggle/working/ResNet18_Checkpoints",
    "zip",
    CHECKPOINT_DIR
)

print("Checkpoint files:")
print(os.listdir(CHECKPOINT_DIR))

display(FileLink("/kaggle/working/ResNet18_Checkpoints.zip"))


In [ ]:
checkpoint = torch.load(
    "/kaggle/working/checkpoints/best_resnet18.pth",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best model loaded")
print("Best epoch:", checkpoint["epoch"])
print("Validation Macro AUC:", checkpoint["macro_auc"])

In [ ]:
model.eval()

test_loss = 0.0
all_targets = []
all_probs = []

with torch.no_grad():
    for images, targets in tqdm(test_loader, desc="Testing ResNet18"):

        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.amp.autocast("cuda"):
            outputs = model(images)
            loss = criterion(outputs, targets)

        probabilities = torch.sigmoid(outputs)

        test_loss += loss.item()
        all_targets.append(targets.cpu().numpy())
        all_probs.append(probabilities.cpu().numpy())

test_loss /= len(test_loader)

y_test = np.concatenate(all_targets)
y_test_prob = np.concatenate(all_probs)

# Overall AUC
test_macro_auc = np.mean([
    roc_auc_score(y_test[:, i], y_test_prob[:, i])
    for i in range(len(LABELS))
])

test_micro_auc = roc_auc_score(
    y_test.ravel(),
    y_test_prob.ravel()
)

print("\n===== RESNET18 TEST RESULTS =====")
print("Test images:", len(y_test))
print("Test Loss:", round(test_loss, 4))
print("Test Macro AUC:", round(test_macro_auc, 4))
print("Test Micro AUC:", round(test_micro_auc, 4))

# Disease-wise AUC
auc_results = []

for i, disease in enumerate(LABELS):
    auc = roc_auc_score(y_test[:, i], y_test_prob[:, i])

    auc_results.append({
        "Disease": disease,
        "Test AUC": round(auc, 4)
    })

auc_df = pd.DataFrame(auc_results)
display(auc_df)

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

# 1. Generate VALIDATION predictions using best Epoch 3 model
model.eval()

val_targets = []
val_probs = []

with torch.no_grad():
    for images, targets in tqdm(val_loader, desc="Validation predictions"):

        images = images.to(device, non_blocking=True)

        with torch.amp.autocast("cuda"):
            outputs = model(images)

        val_targets.append(targets.numpy())
        val_probs.append(torch.sigmoid(outputs).cpu().numpy())

y_val = np.concatenate(val_targets)
y_val_prob = np.concatenate(val_probs)

# 2. Select best F1 threshold for each disease using validation data
thresholds = []

for i in range(len(LABELS)):

    precision, recall, threshold_values = precision_recall_curve(
        y_val[:, i],
        y_val_prob[:, i]
    )

    f1_values = (
        2 * precision[:-1] * recall[:-1]
        / (precision[:-1] + recall[:-1] + 1e-8)
    )

    best_threshold = threshold_values[np.argmax(f1_values)]
    thresholds.append(best_threshold)

thresholds = np.array(thresholds)

# 3. Apply validation-selected thresholds to TEST predictions
y_pred = (y_test_prob >= thresholds.reshape(1, -1)).astype(int)

# 4. Disease-wise metrics
results = []

for i, disease in enumerate(LABELS):

    tn, fp, fn, tp = confusion_matrix(
        y_test[:, i],
        y_pred[:, i],
        labels=[0, 1]
    ).ravel()

    specificity = tn / (tn + fp + 1e-8)

    results.append({
        "Disease": disease,
        "Threshold": round(thresholds[i], 4),
        "Precision": precision_score(
            y_test[:, i], y_pred[:, i], zero_division=0
        ),
        "Sensitivity": recall_score(
            y_test[:, i], y_pred[:, i], zero_division=0
        ),
        "Specificity": specificity,
        "F1 Score": f1_score(
            y_test[:, i], y_pred[:, i], zero_division=0
        ),
        "Accuracy": accuracy_score(
            y_test[:, i], y_pred[:, i]
        )
    })

metrics_df = pd.DataFrame(results)

print("\n===== DISEASE-WISE TEST METRICS =====")
display(metrics_df.round(4))

# 5. Overall metrics
print("\n===== OVERALL TEST METRICS =====")

print("Macro Precision:", round(
    precision_score(y_test, y_pred, average="macro", zero_division=0), 4
))

print("Macro Sensitivity:", round(
    recall_score(y_test, y_pred, average="macro", zero_division=0), 4
))

print("Macro Specificity:", round(
    metrics_df["Specificity"].mean(), 4
))

print("Macro F1:", round(
    f1_score(y_test, y_pred, average="macro", zero_division=0), 4
))

print("Micro Precision:", round(
    precision_score(y_test, y_pred, average="micro", zero_division=0), 4
))

print("Micro Sensitivity:", round(
    recall_score(y_test, y_pred, average="micro", zero_division=0), 4
))

print("Micro F1:", round(
    f1_score(y_test, y_pred, average="micro", zero_division=0), 4
))

# Exact match: all 14 predictions must match
print("Exact Match Accuracy:", round(
    accuracy_score(y_test, y_pred), 4
))

In [21]:
# Save disease-wise metrics
metrics_df.to_csv(
    "/kaggle/working/checkpoints/resnet18_disease_metrics.csv",
    index=False
)

# Save thresholds
threshold_df = pd.DataFrame({
    "Disease": LABELS,
    "Optimal_Validation_Threshold": thresholds
})

threshold_df.to_csv(
    "/kaggle/working/checkpoints/resnet18_thresholds.csv",
    index=False
)

# Save overall results
overall_results = {
    "Test_Loss": test_loss,
    "Test_Macro_AUC": test_macro_auc,
    "Test_Micro_AUC": test_micro_auc,
    "Macro_Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro_Sensitivity": recall_score(y_test, y_pred, average="macro", zero_division=0),
    "Macro_Specificity": metrics_df["Specificity"].mean(),
    "Macro_F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
    "Micro_F1": f1_score(y_test, y_pred, average="micro", zero_division=0),
    "Exact_Match_Accuracy": accuracy_score(y_test, y_pred)
}

pd.DataFrame([overall_results]).to_csv(
    "/kaggle/working/checkpoints/resnet18_overall_metrics.csv",
    index=False
)

# Repackage everything
shutil.make_archive(
    "/kaggle/working/ResNet18_Final_Results",
    "zip",
    CHECKPOINT_DIR
)

print("All evaluation results saved.")
print(os.listdir(CHECKPOINT_DIR))

display(FileLink("/kaggle/working/ResNet18_Final_Results.zip"))

All evaluation results saved.
['resnet18_overall_metrics.csv', 'latest_resnet18.pth', 'best_resnet18.pth', 'training_history.csv', 'resnet18_disease_metrics.csv', 'resnet18_thresholds.csv']


/kaggle/working/ResNet18_Final_Results.zip

In [22]:
import os
import shutil
from IPython.display import display, FileLink

zip_path = "/kaggle/working/ResNet18_Final_Results.zip"

print("File exists:", os.path.exists(zip_path))

if os.path.exists(zip_path):
    print("File size:", round(os.path.getsize(zip_path) / (1024 * 1024), 2), "MB")

    display(FileLink("ResNet18_Final_Results.zip"))
else:
    print("Recreating ZIP...")

    shutil.make_archive(
        "/kaggle/working/ResNet18_Final_Results",
        "zip",
        "/kaggle/working/checkpoints"
    )

    display(FileLink("ResNet18_Final_Results.zip"))

File exists: True
File size: 234.95 MB


/kaggle/working/ResNet18_Final_Results.zip

In [23]:
# Create separate checkpoint folder
RESNET50_DIR = "/kaggle/working/resnet50_checkpoints"
os.makedirs(RESNET50_DIR, exist_ok=True)

# Load pretrained ResNet50
model50 = models.resnet50(
    weights=models.ResNet50_Weights.DEFAULT
)

# Replace final classification layer
model50.fc = nn.Linear(model50.fc.in_features, 14)

model50 = model50.to(device)

# Use both GPUs
if torch.cuda.device_count() > 1:
    model50 = nn.DataParallel(model50)

# Same loss and optimizer configuration
criterion50 = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer50 = optim.AdamW(
    model50.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

scheduler50 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer50,
    mode="max",
    factor=0.5,
    patience=2
)

scaler50 = torch.amp.GradScaler("cuda")

print("Model: ResNet50")
print("Output classes: 14")
print("GPUs:", torch.cuda.device_count())
print("Batch size:", BATCH_SIZE)
print("Checkpoint directory:", RESNET50_DIR)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 205MB/s]


Model: ResNet50
Output classes: 14
GPUs: 2
Batch size: 64
Checkpoint directory: /kaggle/working/resnet50_checkpoints
